# NeuroZip V0 training on Kaggle GPU

This notebook is an orchestration layer, not a second implementation. It discovers the NeuroZip repository uploaded as a Kaggle Dataset, copies it into `/kaggle/working`, and invokes the repository's WikiText preparation and PyTorch training modules.

Enable a Kaggle GPU and Internet access for the first run. The prepared corpus is a deterministic 50 MiB raw-byte window from `wiki.train.raw` plus a 5 MiB validation window from `wiki.valid.raw`.

In [ ]:
from pathlib import Path
import os
import shutil
import sys

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/neurozip')

roots = []
for marker in INPUT_ROOT.rglob('pyproject.toml'):
    candidate = marker.parent
    if (candidate / 'src' / 'neurozip').is_dir() and (candidate / 'docs' / 'DESIGN.md').exists():
        roots.append(candidate)

if not roots:
    raise FileNotFoundError('No NeuroZip Kaggle Dataset found under /kaggle/input')
if len(roots) > 1:
    print('Multiple NeuroZip roots found; using the first sorted candidate:', roots)
SOURCE_ROOT = sorted(roots)[0]
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
shutil.copytree(SOURCE_ROOT, WORK_ROOT)
sys.path.insert(0, str(WORK_ROOT / 'src'))
RUN_ENV = os.environ.copy()
RUN_ENV['PYTHONPATH'] = str(WORK_ROOT / 'src') + os.pathsep + RUN_ENV.get('PYTHONPATH', '')
print('Uploaded source:', SOURCE_ROOT)
print('Working source:', WORK_ROOT)
print('Python path entry:', WORK_ROOT / 'src')

In [ ]:
import json
import subprocess

import torch

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU is required for the real V0 training run; enable Kaggle GPU.')
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

RUN_ROOT = Path('/kaggle/working/neurozip-v0-wikitext103')
DATA_ROOT = RUN_ROOT / 'data'
CACHE_ROOT = Path('/kaggle/working/wikitext-103-cache')
if RUN_ROOT.exists():
    shutil.rmtree(RUN_ROOT)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

prepare_cmd = [
    sys.executable, '-m', 'neurozip.data.prepare_wikitext',
    '--output-dir', str(DATA_ROOT),
    '--cache-dir', str(CACHE_ROOT),
    '--train-bytes', str(50 * 1024 * 1024),
    '--valid-bytes', str(5 * 1024 * 1024),
    '--seed', '20260814',
]
print('Running:', ' '.join(prepare_cmd))
subprocess.run(prepare_cmd, cwd=WORK_ROOT, env=RUN_ENV, check=True)
manifest = json.loads((DATA_ROOT / 'manifest.json').read_text())
print(json.dumps(manifest, indent=2, sort_keys=True))

In [ ]:
config = json.loads((WORK_ROOT / 'configs' / 'v0_wikitext103_kaggle.json').read_text())
train_cfg = config['training']
ARTIFACT_ROOT = RUN_ROOT / 'artifacts'
train_cmd = [
    sys.executable, '-m', 'neurozip.train',
    '--train-path', str(DATA_ROOT / 'train.raw'),
    '--valid-path', str(DATA_ROOT / 'validation.raw'),
    '--output-dir', str(ARTIFACT_ROOT),
    '--steps', str(train_cfg['steps']),
    '--batch-size', str(train_cfg['batch_size']),
    '--sequence-length', str(train_cfg['sequence_length']),
    '--embedding-dim', str(config['model']['embedding_dim']),
    '--hidden-size', str(config['model']['hidden_size']),
    '--num-layers', str(config['model']['num_layers']),
    '--learning-rate', str(train_cfg['learning_rate']),
    '--weight-decay', str(train_cfg['weight_decay']),
    '--gradient-clip', str(train_cfg['gradient_clip']),
    '--eval-every', str(train_cfg['eval_every']),
    '--validation-eval-bytes', str(train_cfg['validation_eval_bytes']),
    '--seed', str(config['dataset']['seed']),
    '--device', 'cuda',
]
print('Running repository training code:')
print(' '.join(train_cmd))
subprocess.run(train_cmd, cwd=WORK_ROOT, env=RUN_ENV, check=True)

In [ ]:
summary = json.loads((ARTIFACT_ROOT / 'summary.json').read_text())
print(json.dumps(summary, indent=2, sort_keys=True))
print('Artifacts:')
for path in sorted(ARTIFACT_ROOT.iterdir()):
    print(f'  {path.name}: {path.stat().st_size:,} bytes')

archive_base = Path('/kaggle/working/neurozip-v0-wikitext103-artifacts')
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=RUN_ROOT)
print('Download this Kaggle output:', archive_path)